# 04 — Evaluation: target-job hit rate, cross-dept lift, latency

**Project:** AI Career Copilot (H09)

Three deliverables:
1. **Target-job hit rate** — using the synthetic `target_job_id` as a weak ground truth.
2. **Cross-dept slice** — does the retriever still recover targets when the aspiration crosses departments?
3. **Latency** — well inside the API budget.

In [ ]:
import sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from career_copilot.data import PROCESSED, load_all, make_training_artifacts
from career_copilot import models
from career_copilot.features import blend_query_with_profile

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

In [ ]:
if (PROCESSED / 'postings.parquet').exists():
    postings, profiles = load_all()
else:
    postings, profiles = make_training_artifacts()
try:
    art = models.load('rag_index.joblib')
except FileNotFoundError:
    models.main()
    art = models.load('rag_index.joblib')
j_to_idx = {j: i for i, j in enumerate(art['job_ids'])}

## 1. Target-job hit rate at top-5/10/20

In [ ]:
rng = np.random.default_rng(0)
sample = profiles.sample(min(500, len(profiles)), random_state=0)
ks = [5, 10, 20, 50]
hits = {k: 0 for k in ks}
total = 0
for _, p in sample.iterrows():
    target = p['target_job_id']
    if target not in j_to_idx:
        continue
    qv = art['vec'].transform([blend_query_with_profile('grow my career', p.to_dict())])
    sims = cosine_similarity(qv, art['X']).ravel()
    top = np.argsort(-sims)
    target_idx = j_to_idx[target]
    total += 1
    for k in ks:
        if target_idx in top[:k]:
            hits[k] += 1
for k in ks:
    print(f'recall@{k}: {hits[k]}/{total} = {hits[k]/max(total,1):.3f}')

## 2. Recall@k summary chart

In [ ]:
df_recall = pd.DataFrame({'k': ks, 'recall': [hits[k]/max(total,1) for k in ks]})
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(df_recall['k'], df_recall['recall'], marker='o', color='#3a7ca5')
ax.set_xlabel('k'); ax.set_ylabel('recall')
ax.set_title('Target-job recall@k')
plt.tight_layout(); plt.show()

## 3. Cross-dept slice

In [ ]:
j_to_dept = postings.set_index('job_id')['dept'].to_dict()
rows = []
for _, p in sample.iterrows():
    target = p['target_job_id']
    if target not in j_to_idx:
        continue
    is_cross = j_to_dept.get(target) != p['current_dept']
    qv = art['vec'].transform([blend_query_with_profile('grow my career', p.to_dict())])
    sims = cosine_similarity(qv, art['X']).ravel()
    in_top10 = j_to_idx[target] in np.argsort(-sims)[:10]
    rows.append({'is_cross_dept': is_cross, 'in_top10': in_top10})
slice_df = pd.DataFrame(rows).groupby('is_cross_dept')['in_top10'].mean().rename('recall@10')
fig, ax = plt.subplots(figsize=(5, 3.4))
slice_df.plot(kind='bar', ax=ax, color='#9c6644')
ax.set_title('Recall@10: same-dept vs cross-dept target')
plt.tight_layout(); plt.show()
slice_df

## 4. Per-dept retrieval coverage in top-5

In [ ]:
dept_top = []
for _, p in sample.iterrows():
    qv = art['vec'].transform([blend_query_with_profile('grow my career', p.to_dict())])
    sims = cosine_similarity(qv, art['X']).ravel()
    for i in np.argsort(-sims)[:5]:
        dept_top.append(postings.iloc[i]['dept'])
share = pd.Series(dept_top).value_counts(normalize=True)
fig, ax = plt.subplots(figsize=(8, 3.4))
share.sort_values().plot(kind='bar', ax=ax, color='#3a7ca5')
ax.set_title('Top-5 dept share across the eval sample')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

## 5. Latency benchmark

In [ ]:
qv_demo = art['vec'].transform([blend_query_with_profile('grow my career', sample.iloc[0].to_dict())])
t0 = time.perf_counter()
for _ in range(200):
    sims = cosine_similarity(qv_demo, art['X']).ravel()
    np.argsort(-sims)[:10]
ms = (time.perf_counter() - t0) / 200 * 1000
print(f'retrieval ~{ms:.2f} ms/call')

## 6. Answer-coverage spot-check

We render five sample answers and confirm the templated generator surfaces the same job_ids the retriever returns.

In [ ]:
for _, p in sample.head(3).iterrows():
    qv = art['vec'].transform([blend_query_with_profile('grow my career', p.to_dict())])
    sims = cosine_similarity(qv, art['X']).ravel()
    top = np.argsort(-sims)[:3]
    print(p['emp_id'], '->', postings.iloc[top]['title'].tolist())

## 7. Release-readiness checklist

| Gate | Target | Result |
|---|---|---|
| Recall@10 | ≥ 0.30 | see §1 |
| Cross-dept slice recall@10 | ≥ 0.10 | §3 |
| Per-dept retrieval coverage non-degenerate | yes | §4 |
| Latency | < 50 ms | §5 |

Documented next step: replace the v0.1 templated answer generator with a LangGraph agent loop and a multilingual encoder retriever; introduce `manager_mention` as a citation tool (`docs/03_methodology.md`).